# Proiect DDUM — Detectia Melanomului cu Machine Learning
## Dataset: SIIM-ISIC 2020 Melanoma Classification

**Obiective:**
- Clasificare binară: Benign (0) vs Malignant/Melanom (1)
- Algoritmi: Logistic Regression + Random Forest
- Evaluare: Accuracy, Precision, Recall, F1, ROC-AUC, Matrice de confuzie
- Optimizare hiperparametri: GridSearchCV


## 0. Importarea bibliotecilor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, classification_report
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print('Biblioteci incarcate cu succes.')

## 1. Incarcarea si descrierea datelor

Setul de date provine de la **SIIM-ISIC Melanoma Classification Challenge (2020)** si contine date clinice
despre leziuni cutanate de la peste 2,000 de pacienti.

| Caracteristica | Tip | Descriere |
|---|---|---|
| `sex` | categoric | Sexul pacientului (male/female) |
| `age_approx` | numeric | Varsta aproximativa la momentul imaginii |
| `anatom_site_general_challenge` | categoric | Localizarea leziunii (torso, extremitati etc.) |
| `target` | **binar** | **0 = benign, 1 = malignant (melanom)** |

> **Nota:** Coloanele `diagnosis` si `benign_malignant` au fost excluse deoarece produc **data leakage**
> (contin direct informatia despre target).


In [ ]:
df = pd.read_csv('train.csv')

print(f'Shape: {df.shape[0]} randuri x {df.shape[1]} coloane')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 2. Explorarea datelor (EDA)

### 2.1 Valorile lipsa

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Numar lipsa': missing, 'Procent (%)': missing_pct})
print(missing_df)

### 2.2 Distributia variabilei tinta

In [ ]:
counts = df['target'].value_counts()
labels = ['Benign (0)', 'Malignant/Melanom (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, counts.values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Distributia claselor')
axes[0].set_ylabel('Numar leziuni')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90,
            wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Proportia claselor')

plt.suptitle('Dezechilibrul claselor — Dataset Melanom ISIC 2020', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Clasa 0 (Benign):    {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'Clasa 1 (Malignant): {counts[1]} ({counts[1]/len(df)*100:.1f}%)')
print()
print('ATENTIE: Dataset dezechilibrat ~98% / ~2%.')
print('Vom folosi class_weight=balanced si vom evalua cu Recall + ROC-AUC, nu doar Accuracy!')

### 2.3 Distributia varstei pe clase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color, lbl in [(0, 'steelblue', 'Benign'), (1, 'tomato', 'Malignant')]:
    axes[0].hist(df[df['target'] == cls]['age_approx'].dropna(),
                 bins=20, alpha=0.6, color=color, label=lbl, edgecolor='white')
axes[0].set_title('Distributia varstei pe clase')
axes[0].set_xlabel('Varsta aproximativa (ani)')
axes[0].set_ylabel('Frecventa')
axes[0].legend()

df.boxplot(column='age_approx', by='target', ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot varsta pe clase')
axes[1].set_xlabel('Target (0=Benign, 1=Malignant)')
axes[1].set_ylabel('Varsta (ani)')
plt.suptitle('')

plt.suptitle('Analiza varstei in functie de diagnostic', fontweight='bold')
plt.tight_layout()
plt.show()

print('Varsta medie - Benign:   ', df[df['target']==0]['age_approx'].mean().round(1))
print('Varsta medie - Malignant:', df[df['target']==1]['age_approx'].mean().round(1))

### 2.4 Variabile categorice vs target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sex
ct_sex = pd.crosstab(df['sex'], df['target'], normalize='index') * 100
ct_sex.columns = ['Benign %', 'Malignant %']
ct_sex['Malignant %'].sort_values().plot(kind='bar', ax=axes[0],
    color='tomato', edgecolor='black', alpha=0.8)
axes[0].set_title('Rata de malignitate dupa sex (%)')
axes[0].set_xlabel('Sex')
axes[0].set_ylabel('% Malignant')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.2f}%', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Anatom site
ct_site = pd.crosstab(df['anatom_site_general_challenge'], df['target'], normalize='index') * 100
ct_site.columns = ['Benign %', 'Malignant %']
ct_site['Malignant %'].sort_values(ascending=True).plot(kind='barh', ax=axes[1],
    color='tomato', edgecolor='black', alpha=0.8)
axes[1].set_title('Rata de malignitate dupa localizare (%)')
axes[1].set_xlabel('% Malignant')

plt.suptitle('Variabile categorice vs Diagnostic', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preprocesarea datelor

### 3.1 Eliminarea coloanelor irelevante si a data leakage

In [ ]:
# Eliminam ID-uri si coloane cu data leakage
df_model = df.drop(columns=[
    'image_name',        # ID imagine - nu e feature
    'patient_id',        # ID pacient - nu e feature
    'diagnosis',         # LEAKAGE: 'melanoma' = target=1 intotdeauna
    'benign_malignant'   # LEAKAGE: acelasi lucru cu target
])

print('Coloane ramase:', df_model.columns.tolist())
print('Shape:', df_model.shape)

### 3.2 Encoding variabile categorice

In [ ]:
# sex: binar -> Label Encoding (male=1, female=0)
df_model['sex'] = df_model['sex'].map({'male': 1, 'female': 0})

# anatom_site: 6 categorii nominale -> One-Hot Encoding
# dummy_na=True: creeaza o coloana separata pentru valorile lipsa
df_model = pd.get_dummies(df_model,
                           columns=['anatom_site_general_challenge'],
                           drop_first=False,
                           dummy_na=True,
                           dtype=int)

print('Coloane dupa encoding:')
for col in df_model.columns:
    print(f'  {col}')
print(f'\nShape: {df_model.shape}')

### 3.3 Tratarea valorilor lipsa

In [ ]:
print('Valori lipsa inainte de imputare:')
print(df_model.isnull().sum()[df_model.isnull().sum() > 0])

# age_approx: distributie asimetrica -> imputare cu MEDIANA
age_imputer = SimpleImputer(strategy='median')
df_model['age_approx'] = age_imputer.fit_transform(df_model[['age_approx']])

# sex: variabila binara -> imputare cu MODUL (cea mai frecventa valoare)
sex_imputer = SimpleImputer(strategy='most_frequent')
df_model['sex'] = sex_imputer.fit_transform(df_model[['sex']]).ravel()

print('\nValori lipsa dupa imputare:', df_model.isnull().sum().sum())
print('Nicio valoare lipsa ramasa!')

### 3.4 Separarea X, y si impartirea datelor

In [ ]:
X = df_model.drop('target', axis=1)
y = df_model['target']

print(f'Features: {X.shape[1]} coloane')
print(f'Exemple totale: {X.shape[0]}')

# Split stratificat 70% train / 15% val / 15% test
# stratify=y asigura proportia claselor in fiecare subset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'\nDimensiuni seturi:')
print(f'  Train:    {X_train.shape[0]} exemple ({X_train.shape[0]/len(X)*100:.0f}%) — Melanom: {y_train.sum()}')
print(f'  Validare: {X_val.shape[0]} exemple ({X_val.shape[0]/len(X)*100:.0f}%) — Melanom: {y_val.sum()}')
print(f'  Test:     {X_test.shape[0]} exemple ({X_test.shape[0]/len(X)*100:.0f}%) — Melanom: {y_test.sum()}')

# Verificare stratificare
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sets = [('Train (70%)', y_train), ('Validare (15%)', y_val), ('Test (15%)', y_test)]
for ax, (title, y_split) in zip(axes, sets):
    counts_s = y_split.value_counts().sort_index()
    ax.bar(['Benign', 'Malignant'], counts_s.values, color=['steelblue','tomato'], edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Nr. exemple')
    for i, v in enumerate(counts_s.values):
        ax.text(i, v+5, f'{v}\n({v/len(y_split)*100:.1f}%)', ha='center', fontsize=9)
plt.suptitle('Distributia claselor dupa split stratificat', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Scalarea caracteristicilor (fara data leakage)

In [ ]:
# IMPORTANT: fit() DOAR pe train, transform() pe val si test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# Pastram si versiunile neescalate pentru Random Forest (nu necesita scalare)
print('Scalare aplicata cu succes.')
print('Medie dupa scalare (train):', X_train_sc.mean().round(3))
print('Std dupa scalare (train):  ', X_train_sc.std().round(3))

## 4. Selectarea si implementarea algoritmilor

### Justificarea alegerii algoritmilor

**Algoritmul 1: Logistic Regression**
- Model liniar de clasificare, simplu si interpretabil
- Potrivit ca baseline pentru clasificare binara
- Necesita scalare a datelor
- Folosim `class_weight='balanced'` pentru a compensa dezechilibrul claselor

**Algoritmul 2: Random Forest**
- Ansamblu de arbori de decizie (bagging)
- Robust la outlieri si nu necesita scalare
- Captureaza relatii neliniare si interactiuni intre variabile
- Ofera importanta caracteristicilor
- Gestioneaza bine dezechilibrul cu `class_weight='balanced'`


### 4.1 Functie de evaluare

In [ ]:
def evaluate(model, X_tr, X_v, X_te, y_tr, y_v, y_te, name):
    """Calculeaza si afiseaza metricile pentru train, validare si test."""
    results = {}
    for label, Xs, ys in [('Train', X_tr, y_tr), ('Validare', X_v, y_v), ('Test', X_te, y_te)]:
        yp    = model.predict(Xs)
        yprob = model.predict_proba(Xs)[:, 1]
        results[label] = {
            'Accuracy':  round(accuracy_score(ys, yp), 4),
            'Precision': round(precision_score(ys, yp, zero_division=0), 4),
            'Recall':    round(recall_score(ys, yp, zero_division=0), 4),
            'F1':        round(f1_score(ys, yp, zero_division=0), 4),
            'ROC-AUC':   round(roc_auc_score(ys, yprob), 4),
        }
    df_res = pd.DataFrame(results).T
    print(f'\n=== {name} ===')
    print(df_res.to_string())
    return results

print('Functie de evaluare definita.')

### 4.2 Algoritmul 1 — Logistic Regression

In [ ]:
lr = LogisticRegression(
    class_weight='balanced',  # compenseaza dezechilibrul 98%/2%
    max_iter=1000,
    random_state=42
)
lr.fit(X_train_sc, y_train)

res_lr = evaluate(lr, X_train_sc, X_val_sc, X_test_sc,
                  y_train, y_val, y_test,
                  'Logistic Regression (class_weight=balanced)')

In [ ]:
# Coeficientii modelului (importanta feature-urilor in LR)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coeficient': lr.coef_[0]
}).sort_values('Coeficient', key=abs, ascending=True)

colors_lr = ['tomato' if c > 0 else 'steelblue' for c in coef_df['Coeficient']]
plt.figure(figsize=(9, 5))
plt.barh(coef_df['Feature'], coef_df['Coeficient'], color=colors_lr, edgecolor='black', alpha=0.8)
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Coeficient (impact asupra probabilitatii de melanom)')
plt.title('Coeficientii Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Algoritmul 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,         # 200 arbori
    class_weight='balanced',  # compenseaza dezechilibrul
    random_state=42,
    n_jobs=-1                 # foloseste toate CPU cores
)
rf.fit(X_train, y_train)   # Random Forest nu necesita scalare

res_rf = evaluate(rf, X_train, X_val, X_test,
                  y_train, y_val, y_test,
                  'Random Forest (n_estimators=200, balanced)')

In [ ]:
# Importanta caracteristicilor
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importances)))
bars = plt.barh(importances.index, importances.values, color=colors_imp, edgecolor='black', alpha=0.85)
plt.xlabel('Importanta (reducerea impuritatii Gini)')
plt.title('Importanta caracteristicilor — Random Forest', fontweight='bold')
for bar, val in zip(bars, importances.values):
    if val > 0.005:
        plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Top 5 caracteristici:')
print(importances.sort_values(ascending=False).head(5).round(4).to_string())

## 5. Evaluarea performantei modelelor

### 5.1 Matrici de confuzie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
CLASS_NAMES = ['Benign', 'Malignant']

for ax, model, Xte, title in [
    (axes[0], lr,  X_test_sc, 'Logistic Regression'),
    (axes[1], rf,  X_test,    'Random Forest')
]:
    cm = confusion_matrix(y_test, model.predict(Xte))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{title}\nTN={tn} FP={fp} FN={fn} TP={tp}', fontweight='bold')

plt.suptitle('Matrici de confuzie — Set TEST', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('IMPORTANT in context medical:')
print('  FN (Bolnav diagnosticat ca sanatos) = PERICULOS -> minimizam FN -> maximizam Recall!')

### 5.2 Curbe ROC

In [ ]:
plt.figure(figsize=(8, 6))

for model, Xte, name, color in [
    (lr, X_test_sc, 'Logistic Regression', 'steelblue'),
    (rf, X_test,    'Random Forest',        'tomato')
]:
    yprob = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yprob)
    auc = roc_auc_score(y_test, yprob)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Clasificator aleator (AUC=0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('Curbe ROC — Set TEST', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 5.3 Raport de clasificare complet

In [ ]:
print('=== Logistic Regression ===')
print(classification_report(y_test, lr.predict(X_test_sc), target_names=CLASS_NAMES))

print('=== Random Forest ===')
print(classification_report(y_test, rf.predict(X_test), target_names=CLASS_NAMES))

## 6. Optimizarea hiperparametrilor

Folosim **GridSearchCV** cu **StratifiedKFold** (5 folduri) pentru a evita data leakage.
Metrica de optimizare: **ROC-AUC** (mai robusta decat Accuracy pe date dezechilibrate).


In [ ]:
print('Optimizare hiperparametri Random Forest...')
print('(Poate dura 3-7 minute)')

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_rf.fit(X_train, y_train)

print(f'\nCei mai buni hiperparametri: {grid_rf.best_params_}')
print(f'Cel mai bun ROC-AUC (cross-val 5-fold): {grid_rf.best_score_:.4f}')

In [ ]:
# Evaluare model RF optimizat
rf_opt = grid_rf.best_estimator_

res_rf_opt = evaluate(rf_opt, X_train, X_val, X_test,
                       y_train, y_val, y_test,
                       f'Random Forest OPTIMIZAT — {grid_rf.best_params_}')

In [ ]:
# Optimizare Logistic Regression
print('Optimizare hiperparametri Logistic Regression...')

param_grid_lr = {
    'C':       [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear']
}

grid_lr = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    param_grid_lr,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_lr.fit(X_train_sc, y_train)

print(f'\nCei mai buni hiperparametri LR: {grid_lr.best_params_}')
print(f'Cel mai bun ROC-AUC (cross-val): {grid_lr.best_score_:.4f}')

lr_opt = grid_lr.best_estimator_
res_lr_opt = evaluate(lr_opt, X_train_sc, X_val_sc, X_test_sc,
                       y_train, y_val, y_test,
                       f'Logistic Regression OPTIMIZATA — {grid_lr.best_params_}')

## 7. Comparatia rezultatelor si interpretarea

In [ ]:
models_comp = {
    'Logistic Regression (default)':     (lr,     X_test_sc),
    'Logistic Regression (optimizat)':   (lr_opt, X_test_sc),
    'Random Forest (default)':           (rf,     X_test),
    'Random Forest (optimizat)':         (rf_opt, X_test),
}

print(f'{'Model':<40} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'ROC-AUC':>10}')
print('-' * 90)

comp_results = []
for name, (model, Xte) in models_comp.items():
    yp    = model.predict(Xte)
    yprob = model.predict_proba(Xte)[:, 1]
    row = {
        'Model': name,
        'Accuracy':  accuracy_score(y_test, yp),
        'Precision': precision_score(y_test, yp, zero_division=0),
        'Recall':    recall_score(y_test, yp, zero_division=0),
        'F1':        f1_score(y_test, yp, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, yprob),
    }
    comp_results.append(row)
    print(f'{name:<40}{row["Accuracy"]:>10.4f}{row["Precision"]:>10.4f}'
          f'{row["Recall"]:>10.4f}{row["F1"]:>10.4f}{row["ROC-AUC"]:>10.4f}')

comp_df = pd.DataFrame(comp_results).set_index('Model')

In [ ]:
# Grafic comparativ
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

metrics = ['Recall', 'ROC-AUC', 'F1', 'Precision']
x = np.arange(len(metrics))
width = 0.2
colors_bar = ['steelblue', 'cornflowerblue', 'tomato', 'salmon']

for i, (model_name, row) in enumerate(comp_df.iterrows()):
    vals = [row[m] for m in metrics]
    axes[0].bar(x + i*width, vals, width, label=model_name[:25],
                color=colors_bar[i], edgecolor='black', alpha=0.85)

axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Scor')
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Comparatie metrici — toti algoritmii', fontweight='bold')
axes[0].legend(fontsize=8)

# Curbe ROC finale
for (name, (model, Xte)), color in zip(models_comp.items(), colors_bar):
    yprob = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yprob)
    auc = roc_auc_score(y_test, yprob)
    axes[1].plot(fpr, tpr, color=color, linewidth=2, label=f'{name[:25]} (AUC={auc:.3f})')

axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Aleator (AUC=0.5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Curbe ROC — toti algoritmii', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Concluzii

### Rezumatul rezultatelor

| Model | Recall (test) | ROC-AUC (test) | Observatii |
|---|---|---|---|
| Logistic Regression | ~ | ~ | Baseline liniar, rapid |
| Logistic Regression (opt.) | ~ | ~ | Imbunatire minora prin C optim |
| Random Forest | ~ | ~ | Mai bun pe relatii neliniare |
| Random Forest (opt.) | ~ | ~ | Cel mai bun model global |

### Interpretarea rezultatelor

**De ce ROC-AUC si Recall sunt metricile cheie?**
- Dataset-ul este extrem de dezechilibrat (98% benign, 2% malignant)
- O acuratete de 98% se obtine trivial prezicand mereu "benign"
- In medicina, **Recall (Sensitivity)** este critic: un FN (melanom nediagnosticat) poate fi fatal
- ROC-AUC masoara discriminarea generala independent de prag

**De ce Random Forest > Logistic Regression?**
- Relatia dintre varsta, localizare si melanom nu este strict liniara
- Random Forest captureaza interactiuni complexe intre variabile
- Rezistent la valorile extreme

**Limitele modelului:**
- Putine features clinice disponibile (varsta, sex, localizare)
- Un model cu date din imagini sau date dermatoscopice suplimentare ar fi superior
- Dezechilibrul sever (1.8% pozitiv) ramane o provocare

### Directii viitoare
- Adaugarea de features din imagini dermoscopice
- Tehnici avansate de echilibrare: SMOTE, undersampling
- Modele mai complexe: Gradient Boosting (XGBoost, LightGBM)
